In [1]:
# Instalacja bibliotek
!pip install boto3 python-dotenv pandas scikit-learn

In [9]:
import boto3
import io

# Połącz z DO Spaces
s3 = boto3.client(
    's3',
    endpoint_url=os.getenv('AWS_ENDPOINT_URL_S3'),
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
)

# Test połączenia - wylistuj pliki
bucket = os.getenv('BUCKET_NAME')
response = s3.list_objects_v2(Bucket=bucket, Prefix='Polmaraton/')
for obj in response.get('Contents', []):
    print(obj['Key'])

Polmaraton/
Polmaraton/41__zadanie_domowe.ipynb
Polmaraton/halfmarathon_wroclaw_2023__final (1).csv
Polmaraton/halfmarathon_wroclaw_2024__final (2).csv


In [10]:
from dotenv import load_dotenv
import os

# Podaj dokładną ścieżkę do .env
load_dotenv(dotenv_path='.env', override=True)

print("BUCKET:", os.getenv('BUCKET_NAME'))
print("ENDPOINT:", os.getenv('AWS_ENDPOINT_URL_S3'))
print("KEY ID:", os.getenv('AWS_ACCESS_KEY_ID'))

BUCKET: od-zera-voland
ENDPOINT: https://fra1.digitaloceanspaces.com
KEY ID: DO00L6ALLUZ9BXQ6N3DQ


In [11]:
def read_csv_from_spaces(s3, bucket, key, sep=';'):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()), sep=sep)

# Wczytaj oba pliki
df_2023 = read_csv_from_spaces(s3, bucket, 'Polmaraton/halfmarathon_wroclaw_2023__final (1).csv')
df_2024 = read_csv_from_spaces(s3, bucket, 'Polmaraton/halfmarathon_wroclaw_2024__final (2).csv')

# Połącz w jeden DataFrame
df = pd.concat([df_2023, df_2024], ignore_index=True)
print(f"Liczba rekordów: {len(df)}")
print(f"Kolumny: {df.columns.tolist()}")

Liczba rekordów: 21957
Kolumny: ['Miejsce', 'Numer startowy', 'Imię', 'Nazwisko', 'Miasto', 'Kraj', 'Drużyna', 'Płeć', 'Płeć Miejsce', 'Kategoria wiekowa', 'Kategoria wiekowa Miejsce', 'Rocznik', '5 km Czas', '5 km Miejsce Open', '5 km Tempo', '10 km Czas', '10 km Miejsce Open', '10 km Tempo', '15 km Czas', '15 km Miejsce Open', '15 km Tempo', '20 km Czas', '20 km Miejsce Open', '20 km Tempo', 'Tempo Stabilność', 'Czas', 'Tempo']


In [12]:
# Czyszczenie danych
def convert_time_to_seconds(time):
    if pd.isnull(time) or time in ['DNS', 'DNF']:
        return None
    time = time.split(':')
    return int(time[0]) * 3600 + int(time[1]) * 60 + int(time[2])

# Zamień czasy na sekundy
df['Czas'] = df['Czas'].apply(convert_time_to_seconds)
df['5 km Czas'] = df['5 km Czas'].apply(convert_time_to_seconds)

# Zamień płeć na liczby
df['Płeć'] = df['Płeć'].map({'M': 0, 'K': 1})

# Wyciągnij wiek z kategorii wiekowej (M20, M30, K40 itd.)
df['Wiek'] = df['Kategoria wiekowa'].str.extract(r'(\d+)').astype(float)

# Usuń wiersze z brakami
df_clean = df[['Płeć', 'Wiek', '5 km Czas', 'Czas']].dropna()

print(f"Rekordów po czyszczeniu: {len(df_clean)}")
print(df_clean.head())

Rekordów po czyszczeniu: 18393
   Płeć  Wiek  5 km Czas    Czas
0   0.0  30.0      877.0  3899.0
1   0.0  30.0      888.0  3983.0
2   0.0  20.0      946.0  4104.0
3   0.0  30.0      971.0  4216.0
4   0.0  20.0      972.0  4227.0


In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

# Podział na cechy i target
X = df_clean[['Płeć', 'Wiek', '5 km Czas']]
y = df_clean['Czas']

# Podział na trening i test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Trenowanie modelu
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Ocena modelu
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.0f} sekund = {mae/60:.1f} minut")

# Zapisz model lokalnie
joblib.dump(model, 'model_polmaraton.pkl')
print("Model zapisany lokalnie!")

MAE: 338 sekund = 5.6 minut
Model zapisany lokalnie!


In [14]:
# Zapisz model do DO Spaces
s3.upload_file(
    'model_polmaraton.pkl',
    bucket,
    'Polmaraton/model_polmaraton.pkl'
)
print("Model zapisany do DO Spaces!")

Model zapisany do DO Spaces!
